# MLflow with Docker

## Topic Goal
Generate Dockerfile for serving same-run model URI.

**Correction applied:** signature and registry now use the **same run** that logs the actual model for the use case.

## 1. Import Libraries

In [ ]:
# Import os to create folders and clean MLflow project environment variables.
import os

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import MLflow for tracking, model logging, and model registry.
import mlflow

# Import sklearn flavor support for logging scikit-learn pipelines.
import mlflow.sklearn

# Import infer_signature to capture model input and output schema.
from mlflow.models.signature import infer_signature

# Import pandas for CSV data processing.
import pandas as pd

# Import NumPy for numerical operations.
import numpy as np

# Import matplotlib for artifact charts.
import matplotlib.pyplot as plt

# Import hashlib for dataset versioning examples.
import hashlib

# Import requests for REST client examples.
import requests

# Import train-test split utility.
from sklearn.model_selection import train_test_split

# Import preprocessing utilities.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Import Pipeline to package preprocessing and model together.
from sklearn.pipeline import Pipeline

# Import model algorithms.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


## 2. Configure MLflow

In [ ]:
EXPERIMENT_NAME = "advanced_11_mlflow_docker"
RUN_NAME = "11_mlflow_docker_same_run_usecase"

# Remove inherited MLflow Project variables to avoid experiment ID issues.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local folders.
os.makedirs("mlruns", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# Configure local tracking store.
mlflow.set_tracking_uri("file:./mlruns")

# Define dataset paths.
DATA_PATH = "datasets/customer_churn_500.csv"
CURRENT_DATA_PATH = "datasets/customer_churn_current_500.csv"

# Set experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name for this topic.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Print setup details.
print("Experiment:", EXPERIMENT_NAME)
print("Registered Model Name:", REGISTERED_MODEL_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())


## 3. Load and Prepare Dataset

In [ ]:
# Load baseline dataset.
df = pd.read_csv(DATA_PATH)

# Display first five records.
display(df.head())

# Define target column.
target_column = "churn"

# Create feature matrix.
X = df.drop(columns=["customer_id", target_column])

# Create target vector.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Create preprocessing transformer.
preprocessor = ColumnTransformer(
    transformers=[
        # Scale numerical features.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical features.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Split data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Print data summary.
print("Shape:", df.shape)
print("Categorical:", categorical_columns)
print("Numerical:", numerical_columns)


## 4. Topic-Specific Setup

In [ ]:
# Dockerfile will be generated after model_uri is created.
with open("outputs/topic_artifact.txt", "w") as file:
    file.write("Docker packages model serving runtime and dependencies.\n")


## 5. Train, Infer Signature, Log Model, and Register from the SAME Run

In [ ]:
# Create model.
model = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)

# Create complete pipeline.
pipeline = Pipeline(steps=[
    # Apply preprocessing.
    ("preprocessor", preprocessor),

    # Train model.
    ("model", model),
])

# Train pipeline.
pipeline.fit(X_train, y_train)

# Predict test data.
predictions = pipeline.predict(X_test)

# Calculate metrics.
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)
f1 = f1_score(y_test, predictions, zero_division=0)

# Create input example for signature.
input_example = X_test.head(5)

# Generate sample predictions for signature.
sample_predictions = pipeline.predict(input_example)

# Infer model signature.
signature = infer_signature(input_example, sample_predictions)
# Start the SAME MLflow run for params, metrics, artifacts, signature, model, and model URI.
with mlflow.start_run(run_name=RUN_NAME):

    # Log dataset path.
    mlflow.log_param("dataset", DATA_PATH)

    # Log topic.
    mlflow.log_param("topic", EXPERIMENT_NAME)

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # Log metrics.
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    # Log any topic-specific artifacts if available.
    if os.path.exists("outputs/topic_artifact.txt"):
        mlflow.log_artifact("outputs/topic_artifact.txt")

    # Log any topic-specific chart if available.
    if os.path.exists("outputs/topic_chart.png"):
        mlflow.log_artifact("outputs/topic_chart.png")

    # Log model WITH signature and input example in the SAME run.
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from this SAME run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Register the model using the SAME run's model URI.
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

# Print results.
print("Same-run model URI:", model_uri)
print("Registered model:", registered_model.name)
print("Registered version:", registered_model.version)
print("F1-score:", f1)


## 6. Topic-Specific Output / Deployment Step

In [ ]:
# Define Docker image and container port examples.
docker_image_name = "churn-mlflow-model"
container_port = 5001

# Print Docker configuration.
print("Docker image name:", docker_image_name)
print("Container port:", container_port)

# Generate Dockerfile using same-run model URI.
dockerfile = f'''
FROM python:3.11-slim
WORKDIR /app
RUN pip install mlflow==2.20.1 scikit-learn pandas numpy
CMD ["mlflow", "models", "serve", "-m", "{model_uri}", "-p", "5001", "--host", "0.0.0.0", "--env-manager", "local"]
'''
with open("outputs/Dockerfile", "w") as file:
    file.write(dockerfile)
print("Generated outputs/Dockerfile")


## Trainer Note

In MLflow UI, open the run named like `*_same_run_usecase`. You will see params, metrics, model artifact, input example, signature, and the registry source all connected to the same run.